In [15]:
"""
Script de limpieza tipo ETL para TIENDA_RETAIL.xlsx
Replica el proceso descrito en el informe (sección 3.4):
 Extraer -> Limpiar/Validar -> Transformar (dimensiones + hechos) -> Cargar (Excel limpio)

Genera:
  - dimProducto   (clave subrogada Producto_SK)
  - dimTienda     (clave subrogada Tienda_SK)
  - dimTiempo     (clave subrogada Fecha_SK)
  - dimContexto   (clave subrogada Contexto_SK)
  - dimPromocion  (clave subrogada Promocion_SK)
  - factVentas    (medidas + FKs a las 5 dimensiones)
  - Calidad_Datos (resumen de validaciones)
"""
import pandas as pd
import numpy as np

# ============================================================
# ÚNICAS DOS LÍNEAS QUE DEBES REVISAR
# ============================================================
RUTA_ORIGEN = r'C:\Users\Usuario\Desktop\CURSOS\CURSO MIERCOLES\TIENDA RETAIL.xlsx'
RUTA_SALIDA = r'C:\Users\Usuario\Desktop\CURSOS\CURSO MIERCOLES\TIENDA RETAIL_LIMPIO.xlsx'
NOMBRE_HOJA = 'TIENDA RETAIL'   # nombre de la pestaña dentro del Excel (sin .xlsx)

# ============================================================
# PASO 1 - EXTRAER
# ============================================================
df = pd.read_excel(RUTA_ORIGEN, sheet_name='TIENDA RETAIL')
filas_originales = len(df)

# ============================================================
# PASO 2 - LIMPIAR Y VALIDAR (equivalente a RF-03 / RNF-02)
# ============================================================
reporte_calidad = {}

nulos_por_columna = df.isnull().sum()
reporte_calidad['nulos_totales'] = int(nulos_por_columna.sum())
df = df.dropna()

duplicados_exactos = df.duplicated().sum()
reporte_calidad['duplicados_exactos_eliminados'] = int(duplicados_exactos)
df = df.drop_duplicates()

Q1 = df['Precio'].quantile(0.25)
Q3 = df['Precio'].quantile(0.75)
IQR = Q3 - Q1
lim_inf, lim_sup = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
mask_outliers = (df['Precio'] < lim_inf) | (df['Precio'] > lim_sup)
reporte_calidad['precios_atipicos_detectados'] = int(mask_outliers.sum())
reporte_calidad['precios_atipicos_pct'] = round(mask_outliers.mean() * 100, 3)
df['Precio_Atipico'] = mask_outliers

combos_producto = df.groupby('Product ID')['Categoria'].nunique()
productos_inconsistentes = combos_producto[combos_producto > 1]
reporte_calidad['product_id_con_categoria_multiple'] = int(len(productos_inconsistentes))

# ============================================================
# PASO 3 - TRANSFORMAR (construir dimensiones con clave subrogada)
# ============================================================
dim_producto = df[['Product ID', 'Categoria']].drop_duplicates().reset_index(drop=True)
dim_producto.insert(0, 'Producto_SK', range(1, len(dim_producto) + 1))

dim_tienda = df[['Store ID', 'Region']].drop_duplicates().reset_index(drop=True)
dim_tienda.insert(0, 'Tienda_SK', range(1, len(dim_tienda) + 1))

dim_tiempo = df[['Fecha', 'Estacionalidad']].drop_duplicates().sort_values('Fecha').reset_index(drop=True)
dim_tiempo['Año'] = dim_tiempo['Fecha'].dt.year
dim_tiempo['Mes'] = dim_tiempo['Fecha'].dt.month
dim_tiempo['Trimestre'] = dim_tiempo['Fecha'].dt.quarter
dim_tiempo.insert(0, 'Fecha_SK', range(1, len(dim_tiempo) + 1))

dim_contexto = df[['Condiciones Climaticas', 'Epidemic']].drop_duplicates().reset_index(drop=True)
dim_contexto.insert(0, 'Contexto_SK', range(1, len(dim_contexto) + 1))

dim_promocion = df[['Promoción']].drop_duplicates().reset_index(drop=True)
dim_promocion['Descripcion'] = dim_promocion['Promoción'].map({0: 'Sin promoción', 1: 'Con promoción'})
dim_promocion.insert(0, 'Promocion_SK', range(1, len(dim_promocion) + 1))

# ============================================================
# PASO 4 - CARGAR (fact table con Lookups a cada dimensión)
# ============================================================
fact = df.merge(dim_producto, on=['Product ID', 'Categoria'], how='left')
fact = fact.merge(dim_tienda, on=['Store ID', 'Region'], how='left')
fact = fact.merge(dim_tiempo[['Fecha', 'Fecha_SK']], on='Fecha', how='left')
fact = fact.merge(dim_contexto, on=['Condiciones Climaticas', 'Epidemic'], how='left')
fact = fact.merge(dim_promocion[['Promoción', 'Promocion_SK']], on='Promoción', how='left')

fk_cols = ['Producto_SK', 'Tienda_SK', 'Fecha_SK', 'Contexto_SK', 'Promocion_SK']
filas_sin_match = fact[fk_cols].isnull().any(axis=1).sum()
reporte_calidad['filas_sin_match_en_lookup'] = int(filas_sin_match)

fact_ventas = fact[[
    'Producto_SK', 'Tienda_SK', 'Fecha_SK', 'Contexto_SK', 'Promocion_SK',
    'Inventory Level', 'Unidad Vendida', 'Unidades Ordenadas', 'Precio',
    'Descuento', 'Precios de la competencia', 'Demanda', 'Precio_Atipico'
]].rename(columns={
    'Inventory Level': 'Nivel_Inventario',
    'Unidad Vendida': 'Unidad_Vendida',
    'Unidades Ordenadas': 'Unidades_Ordenadas',
    'Precios de la competencia': 'Precio_Competencia',
})

reporte_calidad['filas_originales'] = filas_originales
reporte_calidad['filas_finales_factventas'] = len(fact_ventas)
reporte_calidad['productos_unicos_dimProducto'] = len(dim_producto)
reporte_calidad['tiendas_unicas_dimTienda'] = len(dim_tienda)
reporte_calidad['fechas_unicas_dimTiempo'] = len(dim_tiempo)

df_calidad = pd.DataFrame(list(reporte_calidad.items()), columns=['Indicador', 'Valor'])

# ============================================================
# EXPORTAR A EXCEL
# ============================================================
with pd.ExcelWriter(RUTA_SALIDA, engine='openpyxl') as writer:
    df_calidad.to_excel(writer, sheet_name='Calidad_Datos', index=False)
    dim_producto.to_excel(writer, sheet_name='dimProducto', index=False)
    dim_tienda.to_excel(writer, sheet_name='dimTienda', index=False)
    dim_tiempo.to_excel(writer, sheet_name='dimTiempo', index=False)
    dim_contexto.to_excel(writer, sheet_name='dimContexto', index=False)
    dim_promocion.to_excel(writer, sheet_name='dimPromocion', index=False)
    fact_ventas.to_excel(writer, sheet_name='factVentas', index=False)

print("Reporte de calidad:")
for k, v in reporte_calidad.items():
    print(f"  {k}: {v}")
print("\nArchivo generado en:", RUTA_SALIDA)

Reporte de calidad:
  nulos_totales: 0
  duplicados_exactos_eliminados: 0
  precios_atipicos_detectados: 70
  precios_atipicos_pct: 0.092
  product_id_con_categoria_multiple: 20
  filas_sin_match_en_lookup: 0
  filas_originales: 76000
  filas_finales_factventas: 76000
  productos_unicos_dimProducto: 64
  tiendas_unicas_dimTienda: 5
  fechas_unicas_dimTiempo: 760

Archivo generado en: C:\Users\Usuario\Desktop\CURSOS\CURSO MIERCOLES\TIENDA RETAIL_LIMPIO.xlsx
